In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================
# 1. Imports
# =========================
import os
import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# =========================
# 2. LOAD DATASET
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/xgboost-devign26"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        if f.endswith(".csv"):
            path = os.path.join(root, f)
            print("✅ Loaded:", path)
            df = pd.read_csv(path)
            break
    if df is not None:
        break

if df is None:
    raise Exception("❌ Dataset not found")

print("\n📊 Shape:", df.shape)

# =========================
# 3. PREPROCESS
# =========================
df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)

# =========================
# 4. TRAIN-VAL-TEST SPLIT (Fixing Data Leakage)
# =========================
# First split: 80% Train, 20% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"].astype(str),
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: Divides the Temp into 10% Validation, 10% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# =========================
# 5. DUAL FEATURE ENGINEERING (Word + Char)
# =========================
print("\n⚙️ Vectorizing Data (Dual Word+Char N-Grams)...")

# 5A. Word-Level Vectorizer (Syntax & Keywords)
vec_word = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    # 🔥 UPGRADE: Captures exact syntax and operators (==, {, >)
    token_pattern=r'[a-zA-Z0-9_]+|[^\w\s]', 
    dtype=np.float32 # 🔥 UPGRADE: Cuts RAM usage by 50%
)

# 5B. Char-Level Vectorizer (Motifs, structural chunks, and typos)
vec_char = TfidfVectorizer(
    max_features=15000,
    analyzer='char_wb',
    ngram_range=(3, 5), 
    min_df=2,
    max_df=0.90,
    dtype=np.float32
)

# Fit exclusively on Training Data
print("Fitting vectorizers to Training data...")
X_tr_w = vec_word.fit_transform(X_train)
X_tr_c = vec_char.fit_transform(X_train)

# Stack features horizontally into one dense matrix
X_train_vec = hstack([X_tr_w, X_tr_c]).tocsr()

# Transform Validation & Test Sets safely
X_val_vec  = hstack([vec_word.transform(X_val), vec_char.transform(X_val)]).tocsr()
X_test_vec = hstack([vec_word.transform(X_test), vec_char.transform(X_test)]).tocsr()

print(f"Total Combined Features: {X_train_vec.shape[1]}")

# =========================
# 6. CLASS IMBALANCE HANDLING
# =========================
# Provide a balance guide for XGBoost
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Scale Pos Weight calculated as: {pos_weight:.2f}")

# =========================
# 7. XGBOOST MODEL
# =========================
print("\n🚀 Training XGBoost Classifier...")
model = XGBClassifier(
    n_estimators=400,          # Increased trees for better learning capacity
    learning_rate=0.05,        # Lowered LR pairs with higher estimators to prevent overfitting
    max_depth=7,               # Deeper trees capture complex "if A and B" code rules
    subsample=0.8,
    colsample_bytree=0.8,      # Stops trees from overly relying on a single loud keyword
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_vec, y_train)

# =========================
# 8. MACRO-F1 THRESHOLD TUNING (ON VAL SET)
# =========================
val_probs = model.predict_proba(X_val_vec)[:, 1]

best_macro_f1 = 0
best_t = 0.5

print("\n🔍 Threshold tuning strictly on Validation Set:")

# 🔥 UPGRADE: Instantly fast threshold loop using raw math + optimized for Macro-F1
for t in np.arange(0.20, 0.82, 0.02):
    preds = (val_probs > t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    # Macro-F1 prevents the model from collapsing and predicting only one class
    macro_f1 = (f1_0 + f1_1) / 2

    # Clean printing
    if round(t * 100) % 10 == 0:
        print(f"t={t:.2f} → Macro_F1={macro_f1:.4f} | R0={recall_0:.2f}, R1={recall_1:.2f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_t = t

print(f"\nBest threshold found: {best_t:.2f}")

# =========================
# 9. FINAL EVALUATION (ON TEST SET)
# =========================
print("\nEvaluating untouched Test Set...")
test_probs = model.predict_proba(X_test_vec)[:, 1]
final_preds = (test_probs > best_t).astype(int)

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 10. SAVE RESULTS
# =========================
label_key = "1" if "1" in report else[k for k in report.keys() if str(k).startswith("1")][0]

results = {
    "Model": "XGBoost_Dual_TFIDF",
    "Dataset": "Devign",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/xgb_devign_results.csv", index=False)

print("\n✅ Results saved in /kaggle/working/")

✅ Loaded: /kaggle/input/datasets/nikunjnawal009/xgboost-devign26/Devignx_validation.csv

📊 Shape: (2732, 2)
Train size: 2185 | Val size: 273 | Test size: 274

⚙️ Vectorizing Data (Dual Word+Char N-Grams)...
Fitting vectorizers to Training data...
Total Combined Features: 21771
Scale Pos Weight calculated as: 1.30

🚀 Training XGBoost Classifier...

🔍 Threshold tuning strictly on Validation Set:
t=0.20 → Macro_F1=0.4991 | R0=0.28, R1=0.83
t=0.30 → Macro_F1=0.5465 | R0=0.42, R1=0.72
t=0.40 → Macro_F1=0.5962 | R0=0.57, R1=0.63
t=0.50 → Macro_F1=0.5804 | R0=0.68, R1=0.48
t=0.60 → Macro_F1=0.5214 | R0=0.78, R1=0.29
t=0.70 → Macro_F1=0.4656 | R0=0.87, R1=0.16
t=0.80 → Macro_F1=0.4027 | R0=0.94, R1=0.06

Best threshold found: 0.44

Evaluating untouched Test Set...

✅ Accuracy: 0.5474452554744526

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.51      0.56       155
           1       0.48      0.60      0.53       119

    accura